In [35]:
import pandas as pd

df = pd.read_csv("../data/atlantic_spain_cleaned.csv")
df["date"] = pd.to_datetime(df["date"])

lifecycle = pd.read_csv(
    "../data/lifecycle_features.csv",
    index_col="song_id",
    parse_dates=["entry_date", "exit_date"]
)

lifecycle.head()

,entry_date,exit_date,days_on_playlist,best_rank,worst_rank,avg_rank,peak_popularity,avg_popularity
song_id,,,,,,,,
0 confianza_rvfv,2025-06-02,2025-06-23,22,23,42,29.650000,62,57.550000
"1 de enero, puntacana_rels b",2025-04-28,2025-04-28,1,45,45,45.000000,65,65.000000
1000cosas_lola indigo & manuel turizo,2025-03-10,2025-11-13,249,3,44,20.428571,81,76.171429
100xciento_foreign teck & eladio carrion & de la ghetto,2025-04-30,2025-05-15,16,13,42,28.181818,75,73.727273
14 febreros_quevedo,2025-11-24,2025-11-27,4,9,16,13.500000,68,65.750000


In [36]:
rank_std = (
    df.groupby("song_id")["position"]
      .std()
      .rename("rank_volatility")
)

lifecycle = lifecycle.join(rank_std)

In [37]:
lifecycle.head()

,entry_date,exit_date,days_on_playlist,best_rank,worst_rank,avg_rank,peak_popularity,avg_popularity,rank_volatility
song_id,,,,,,,,,
0 confianza_rvfv,2025-06-02,2025-06-23,22,23,42,29.650000,62,57.550000,5.659412
"1 de enero, puntacana_rels b",2025-04-28,2025-04-28,1,45,45,45.000000,65,65.000000,NaN
1000cosas_lola indigo & manuel turizo,2025-03-10,2025-11-13,249,3,44,20.428571,81,76.171429,9.585679
100xciento_foreign teck & eladio carrion & de la ghetto,2025-04-30,2025-05-15,16,13,42,28.181818,75,73.727273,9.086453
14 febreros_quevedo,2025-11-24,2025-11-27,4,9,16,13.500000,68,65.750000,3.316625


In [38]:
playlist_entries = (
    df.groupby('song_id')
      .size()
      .rename('playlist_entries')
)

lifecycle = lifecycle.join(playlist_entries)

In [39]:
peak_day = (
    df.loc[df.groupby('song_id')['position'].idxmin()]
      [['song_id', 'date']]
      .rename(columns={'date': 'peak_day'})
      .set_index('song_id')
)

lifecycle = lifecycle.join(peak_day)

In [40]:
lifecycle['days_to_peak'] = (
    lifecycle['peak_day']
    - lifecycle['entry_date']
).dt.days

In [41]:
initial_rank = (
    df.sort_values('date')
      .groupby('song_id')
      .first()['position']
      .rename('initial_rank')
)

lifecycle = lifecycle.join(initial_rank)

In [42]:
lifecycle['rank_improvement'] = (
    lifecycle['initial_rank']
    - lifecycle['best_rank']
)

In [43]:
lifecycle.head()

,entry_date,exit_date,days_on_playlist,best_rank,worst_rank,avg_rank,peak_popularity,avg_popularity,rank_volatility,playlist_entries,peak_day,days_to_peak,initial_rank,rank_improvement
song_id,,,,,,,,,,,,,,
0 confianza_rvfv,2025-06-02,2025-06-23,22,23,42,29.650000,62,57.550000,5.659412,20,2025-06-07,5,40,17
"1 de enero, puntacana_rels b",2025-04-28,2025-04-28,1,45,45,45.000000,65,65.000000,NaN,1,2025-04-28,0,45,0
1000cosas_lola indigo & manuel turizo,2025-03-10,2025-11-13,249,3,44,20.428571,81,76.171429,9.585679,245,2025-03-10,0,3,0
100xciento_foreign teck & eladio carrion & de la ghetto,2025-04-30,2025-05-15,16,13,42,28.181818,75,73.727273,9.086453,11,2025-05-11,11,42,29
14 febreros_quevedo,2025-11-24,2025-11-27,4,9,16,13.500000,68,65.750000,3.316625,4,2025-11-24,0,9,0


In [44]:
lifecycle['stability_score'] = (
    lifecycle['days_on_playlist']
    /
    (lifecycle['rank_volatility'] + 1)
)

lifecycle.head()

,entry_date,exit_date,days_on_playlist,best_rank,worst_rank,avg_rank,peak_popularity,avg_popularity,rank_volatility,playlist_entries,peak_day,days_to_peak,initial_rank,rank_improvement,stability_score
song_id,,,,,,,,,,,,,,,
0 confianza_rvfv,2025-06-02,2025-06-23,22,23,42,29.650000,62,57.550000,5.659412,20,2025-06-07,5,40,17,3.303595
"1 de enero, puntacana_rels b",2025-04-28,2025-04-28,1,45,45,45.000000,65,65.000000,NaN,1,2025-04-28,0,45,0,NaN
1000cosas_lola indigo & manuel turizo,2025-03-10,2025-11-13,249,3,44,20.428571,81,76.171429,9.585679,245,2025-03-10,0,3,0,23.522345
100xciento_foreign teck & eladio carrion & de la ghetto,2025-04-30,2025-05-15,16,13,42,28.181818,75,73.727273,9.086453,11,2025-05-11,11,42,29,1.586286
14 febreros_quevedo,2025-11-24,2025-11-27,4,9,16,13.500000,68,65.750000,3.316625,4,2025-11-24,0,9,0,0.926650


In [45]:
df = df.sort_values(["song_id", "date"])

In [46]:
df["daily_rank_change"] = (
    df.groupby("song_id")["position"]
      .diff()
)

In [47]:
df.head(20)

,date,position,song,artist,popularity,duration_ms,album_type,total_tracks,is_explicit,album_cover_url,song_id,duration_min,daily_rank_change
18974,2025-06-02,40,0 CONFIANZA,Rvfv,35,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,NaN
19115,2025-06-05,31,0 CONFIANZA,Rvfv,57,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,-9.0
19159,2025-06-06,25,0 CONFIANZA,Rvfv,57,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,-6.0
19207,2025-06-07,23,0 CONFIANZA,Rvfv,57,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,-2.0
19257,2025-06-08,23,0 CONFIANZA,Rvfv,57,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,0.0
19313,2025-06-09,29,0 CONFIANZA,Rvfv,57,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,6.0
19362,2025-06-10,28,0 CONFIANZA,Rvfv,58,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,-1.0
19412,2025-06-11,28,0 CONFIANZA,Rvfv,58,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,0.0
19460,2025-06-12,26,0 CONFIANZA,Rvfv,58,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,-2.0
19510,2025-06-13,26,0 CONFIANZA,Rvfv,58,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,0.0


In [48]:
avg_daily_change = (
    df.groupby("song_id")["daily_rank_change"]
      .apply(lambda x: x.abs().mean())
      .rename("avg_daily_rank_change")
)

lifecycle = lifecycle.join(avg_daily_change)

In [49]:
df.head()

,date,position,song,artist,popularity,duration_ms,album_type,total_tracks,is_explicit,album_cover_url,song_id,duration_min,daily_rank_change
18974,2025-06-02,40,0 CONFIANZA,Rvfv,35,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,NaN
19115,2025-06-05,31,0 CONFIANZA,Rvfv,57,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,-9.0
19159,2025-06-06,25,0 CONFIANZA,Rvfv,57,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,-6.0
19207,2025-06-07,23,0 CONFIANZA,Rvfv,57,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,-2.0
19257,2025-06-08,23,0 CONFIANZA,Rvfv,57,217500,album,20,False,https://i.scdn.co/image/ab67616d0000b273c172e8...,0 confianza_rvfv,3.625,0.0


In [50]:
lifecycle["rank_volatility"] = (
    lifecycle["rank_volatility"]
    .fillna(0)
)

In [51]:
lifecycle["stability_score"] = (
    lifecycle["days_on_playlist"] /
    (lifecycle["rank_volatility"] + 1)
)

In [52]:
print(lifecycle.columns)

Index(['entry_date', 'exit_date', 'days_on_playlist', 'best_rank',
       'worst_rank', 'avg_rank', 'peak_popularity', 'avg_popularity',
       'rank_volatility', 'playlist_entries', 'peak_day', 'days_to_peak',
       'initial_rank', 'rank_improvement', 'stability_score',
       'avg_daily_rank_change'],
      dtype='str')


In [53]:
print(lifecycle.shape)

(575, 16)


In [54]:
lifecycle.head()

,entry_date,exit_date,days_on_playlist,best_rank,worst_rank,avg_rank,peak_popularity,avg_popularity,rank_volatility,playlist_entries,peak_day,days_to_peak,initial_rank,rank_improvement,stability_score,avg_daily_rank_change
song_id,,,,,,,,,,,,,,,,
0 confianza_rvfv,2025-06-02,2025-06-23,22,23,42,29.650000,62,57.550000,5.659412,20,2025-06-07,5,40,17,3.303595,2.210526
"1 de enero, puntacana_rels b",2025-04-28,2025-04-28,1,45,45,45.000000,65,65.000000,0.000000,1,2025-04-28,0,45,0,1.000000,NaN
1000cosas_lola indigo & manuel turizo,2025-03-10,2025-11-13,249,3,44,20.428571,81,76.171429,9.585679,245,2025-03-10,0,3,0,23.522345,1.102459
100xciento_foreign teck & eladio carrion & de la ghetto,2025-04-30,2025-05-15,16,13,42,28.181818,75,73.727273,9.086453,11,2025-05-11,11,42,29,1.586286,11.700000
14 febreros_quevedo,2025-11-24,2025-11-27,4,9,16,13.500000,68,65.750000,3.316625,4,2025-11-24,0,9,0,0.926650,2.333333


In [55]:
lifecycle.reset_index(inplace=True)

In [56]:
lifecycle["rank_range"] = (
    lifecycle["worst_rank"] -
    lifecycle["best_rank"]
)

In [57]:
lifecycle["popularity_gap"] = (
    lifecycle["peak_popularity"] -
    lifecycle["avg_popularity"]
)

In [58]:
lifecycle.head()

,song_id,entry_date,exit_date,days_on_playlist,best_rank,worst_rank,avg_rank,peak_popularity,avg_popularity,rank_volatility,playlist_entries,peak_day,days_to_peak,initial_rank,rank_improvement,stability_score,avg_daily_rank_change,rank_range,popularity_gap
0,0 confianza_rvfv,2025-06-02,2025-06-23,22,23,42,29.650000,62,57.550000,5.659412,20,2025-06-07,5,40,17,3.303595,2.210526,19,4.450000
1,"1 de enero, puntacana_rels b",2025-04-28,2025-04-28,1,45,45,45.000000,65,65.000000,0.000000,1,2025-04-28,0,45,0,1.000000,NaN,0,0.000000
2,1000cosas_lola indigo & manuel turizo,2025-03-10,2025-11-13,249,3,44,20.428571,81,76.171429,9.585679,245,2025-03-10,0,3,0,23.522345,1.102459,41,4.828571
3,100xciento_foreign teck & eladio carrion & de ...,2025-04-30,2025-05-15,16,13,42,28.181818,75,73.727273,9.086453,11,2025-05-11,11,42,29,1.586286,11.700000,29,1.272727
4,14 febreros_quevedo,2025-11-24,2025-11-27,4,9,16,13.500000,68,65.750000,3.316625,4,2025-11-24,0,9,0,0.926650,2.333333,7,2.250000


In [59]:
lifecycle["rank_range"] = (
    lifecycle["worst_rank"] - lifecycle["best_rank"]
)

lifecycle["popularity_gap"] = (
    lifecycle["peak_popularity"] - lifecycle["avg_popularity"]
)

In [60]:
lifecycle.head()

,song_id,entry_date,exit_date,days_on_playlist,best_rank,worst_rank,avg_rank,peak_popularity,avg_popularity,rank_volatility,playlist_entries,peak_day,days_to_peak,initial_rank,rank_improvement,stability_score,avg_daily_rank_change,rank_range,popularity_gap
0,0 confianza_rvfv,2025-06-02,2025-06-23,22,23,42,29.650000,62,57.550000,5.659412,20,2025-06-07,5,40,17,3.303595,2.210526,19,4.450000
1,"1 de enero, puntacana_rels b",2025-04-28,2025-04-28,1,45,45,45.000000,65,65.000000,0.000000,1,2025-04-28,0,45,0,1.000000,NaN,0,0.000000
2,1000cosas_lola indigo & manuel turizo,2025-03-10,2025-11-13,249,3,44,20.428571,81,76.171429,9.585679,245,2025-03-10,0,3,0,23.522345,1.102459,41,4.828571
3,100xciento_foreign teck & eladio carrion & de ...,2025-04-30,2025-05-15,16,13,42,28.181818,75,73.727273,9.086453,11,2025-05-11,11,42,29,1.586286,11.700000,29,1.272727
4,14 febreros_quevedo,2025-11-24,2025-11-27,4,9,16,13.500000,68,65.750000,3.316625,4,2025-11-24,0,9,0,0.926650,2.333333,7,2.250000


In [61]:
lifecycle = lifecycle[
[
    "song_id",
    "entry_date",
    "exit_date",
    "days_on_playlist",

    "best_rank",
    "worst_rank",
    "avg_rank",
    "initial_rank",

    "rank_improvement",
    "rank_range",
    "rank_volatility",
    "avg_daily_rank_change",

    "peak_popularity",
    "avg_popularity",
    "popularity_gap",

    "playlist_entries",
    "days_to_peak",
    "stability_score"
]
]

In [62]:
print(lifecycle.index[:10])

RangeIndex(start=0, stop=10, step=1)


In [63]:
lifecycle.head()

,song_id,entry_date,exit_date,days_on_playlist,best_rank,worst_rank,avg_rank,initial_rank,rank_improvement,rank_range,rank_volatility,avg_daily_rank_change,peak_popularity,avg_popularity,popularity_gap,playlist_entries,days_to_peak,stability_score
0,0 confianza_rvfv,2025-06-02,2025-06-23,22,23,42,29.650000,40,17,19,5.659412,2.210526,62,57.550000,4.450000,20,5,3.303595
1,"1 de enero, puntacana_rels b",2025-04-28,2025-04-28,1,45,45,45.000000,45,0,0,0.000000,NaN,65,65.000000,0.000000,1,0,1.000000
2,1000cosas_lola indigo & manuel turizo,2025-03-10,2025-11-13,249,3,44,20.428571,3,0,41,9.585679,1.102459,81,76.171429,4.828571,245,0,23.522345
3,100xciento_foreign teck & eladio carrion & de ...,2025-04-30,2025-05-15,16,13,42,28.181818,42,29,29,9.086453,11.700000,75,73.727273,1.272727,11,11,1.586286
4,14 febreros_quevedo,2025-11-24,2025-11-27,4,9,16,13.500000,9,0,7,3.316625,2.333333,68,65.750000,2.250000,4,0,0.926650


In [64]:
lifecycle.to_csv(
    "../data/lifecycle_advanced_features.csv",
    index=True
)